# E11.5: source clustering

The original count files are the inputs. Cell 3 reads the unchanged reviewed membership and gene metadata from the readable metadata files; these contain no expression matrix or embedding. Analysis and figure functions are included in cell 2. No project Python module is required. Set the paths for the stage cells TSV, stage review JSON, and shared common genes TSV in cell 3.

The source pools, retained cells, feature sets, normalization, parameters, annotations and figure styles reproduce the verified result.


In [ ]:
#1. =========== Paths, runtime and imports
from pathlib import Path
import os, sys
# Set the working root before replacing each individual path placeholder.
ROOT = Path(".")
# Set each original downloaded raw-count CSV independently; keys retain the source library identities.
EARLY_RAW_COUNT_FILES = {
    # Set GSM5401073_E11_5_1_raw_count.csv.gz, the original GSM5401073 raw-count input.
    "GSM5401073": ROOT,
    # Set GSM5401074_E11_5_2_raw_count.csv.gz, the original GSM5401074 raw-count input.
    "GSM5401074": ROOT,
    # Set GSM5401075_E11_5_3_raw_count.csv.gz, the original GSM5401075 raw-count input.
    "GSM5401075": ROOT,
}
# Set the stage integration-input output directory; this notebook generates the retained raw-count H5AD.
DATA_DIR = ROOT
# Set the figure output or accepted-reference directory path.
FIGURE_DIR = ROOT
# Set the figure output or accepted-reference directory path.
REFERENCE_DIR = ROOT
# Set the temporary working directory or file path.
TASK_TMP = ROOT
RANDOM_SEED, N_THREADS, FIGURE_DPI = (0, 4, 600)
for key, leaf in [('NUMBA_CACHE_DIR', 'n'), ('MPLCONFIGDIR', 'm'), ('TMP', 't'), ('TEMP', 't')]:
    # Set the temporary working directory or file path.
    folder = ROOT
    folder.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(folder)
for key in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMBA_NUM_THREADS', 'LOKY_MAX_CPU_COUNT'):
    os.environ[key] = str(N_THREADS)
sys.dont_write_bytecode = True
import json, hashlib
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
from scipy import sparse
from threadpoolctl import threadpool_limits
from importlib.metadata import version
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image
from PIL import Image as PILImage
from io import BytesIO
sc.settings.verbosity = 0

from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.colors import Normalize
import matplotlib.patheffects as pe
import textwrap
import csv
import gzip
from scipy.stats import median_abs_deviation
from collections.abc import Mapping
import uuid

In [ ]:
#2. =========== Source-specific analysis and figure functions
UMAP_STYLE = dict(point_size=1.35, point_alpha=0.82, label_fontsize=6.75, legend_fontsize=6.8, label_stroke=1.45, legend_figsize=(10.3, 6.4), labels_figsize=(8.2, 8.0), connector_distance=12, connector_width=0.42, connector_alpha=0.7)
DOTPLOT_STYLE = dict(max_dot_area=36, column_gap_ratio=0.5, gene_font=6.3, row_font=6.0, row_pitch=0.155, gene_rotation=55, color_min=-1.0, color_max=1.0, split_max_genes=36)
DISPLAY_TYPES = {'Otic neurogenic / CVG candidate': 'Otic neurogenic candidate', 'O-2: delaminating otic/CVG neuroblast': 'Delaminating otic/CVG neuroblast', 'O-3: developing CVG neuron (not SGN-specific)': 'Developing CVG neuron', 'Undifferentiated / proliferative CVG': 'Proliferative CVG', 'CVG': 'CVG', 'Undifferentiated SGN': 'Undifferentiated SGN', 'Type Ia + Type II precursor candidate': 'Ia/II precursor candidate', 'Type Ib + Type Ic precursor candidate': 'Ib/Ic precursor candidate', 'Immature Type Ia SGN': 'Immature Ia SGN', 'Immature Type Ib SGN': 'Immature Ib SGN', 'Immature Type Ic SGN': 'Immature Ic SGN', 'Immature Type II SGN': 'Immature II SGN'}
GROUP_NAMES = {'Transition candidates': 'Developmental regulators', 'Delamination / differentiation': 'Delamination', 'SGN differentiation': 'SGN differentiation', 'Additional synaptic function': 'Synaptic function', 'Additional cell cycle': 'Replication', 'Apex-associated program': 'Apex program', 'Exclusion: VGN candidates': 'VGN', 'Exclusion: geniculate / visceral': 'GG / visceral', 'Exclusion: CNS candidates': 'CNS', 'Exclusion: hair cells': 'Hair cells', 'Exclusion: glia': 'Glia', 'Exclusion: mesenchyme': 'Mesenchyme', 'Exclusion: epithelial': 'Epithelium', 'Exclusion: immune': 'Immune', 'Exclusion: erythroid': 'Erythroid'}
CURATED_CATEGORY_ORDER = ['E9.5 | CVG neuroblast', 'E10.5 | O-2: delaminating otic/CVG neuroblast', 'E10.5 | O-3: developing CVG neuron (not SGN-specific)', 'E11.5 | Undifferentiated / proliferative CVG', 'E11.5 | Differentiating SGN', 'E13.5 | Late CVG', 'E13.5 | Differentiating SGN', 'E14.5 | Type Ia + Type II precursor candidate', 'E14.5 | Type Ib + Type Ic precursor candidate', 'E14.5 | Undifferentiated SGN', 'E15.5 | Type Ia + Type II precursor candidate', 'E15.5 | Type Ib + Type Ic precursor candidate', 'E15.5 | Undifferentiated SGN', 'E16.5 | Emerging Type Ic', 'E16.5 | Type Ia + Type II precursor candidate', 'E16.5 | Type Ib + Type Ic precursor candidate', 'E16.5 | Undifferentiated SGN', 'E17.5 | Type Ib + Type Ic precursor candidate', 'E17.5 | Type Ia + Type II precursor candidate', 'E18.5 | Immature Type II SGN', 'E18.5 | Immature Type Ia SGN', 'E18.5 | Immature Type Ib SGN', 'E18.5 | Immature Type Ic SGN', 'E18.5 | Undifferentiated SGN']
CURATED_PALETTE = {'E9.5 | CVG neuroblast': '#b08c43', 'E10.5 | O-2: delaminating otic/CVG neuroblast': '#e08bbd', 'E10.5 | O-3: developing CVG neuron (not SGN-specific)': '#5578e0', 'E11.5 | Undifferentiated / proliferative CVG': '#438cb0', 'E11.5 | Differentiating SGN': '#e08f55', 'E13.5 | Late CVG': '#e05578', 'E13.5 | Differentiating SGN': '#4dc9c9', 'E14.5 | Type Ia + Type II precursor candidate': '#b06d78', 'E14.5 | Type Ib + Type Ic precursor candidate': '#6db08f', 'E14.5 | Undifferentiated SGN': '#79b043', 'E15.5 | Type Ia + Type II precursor candidate': '#8ac97d', 'E15.5 | Type Ib + Type Ic precursor candidate': '#b0946d', 'E15.5 | Undifferentiated SGN': '#d98be0', 'E16.5 | Emerging Type Ic': '#e06c55', 'E16.5 | Type Ia + Type II precursor candidate': '#c94d95', 'E16.5 | Type Ib + Type Ic precursor candidate': '#8b92e0', 'E16.5 | Undifferentiated SGN': '#e0b255', 'E17.5 | Type Ib + Type Ic precursor candidate': '#b0b043', 'E17.5 | Type Ia + Type II precursor candidate': '#e0998b', 'E18.5 | Immature Type II SGN': '#b06743', 'E18.5 | Immature Type Ia SGN': '#9ab06d', 'E18.5 | Immature Type Ib SGN': '#9f65c9', 'E18.5 | Immature Type Ic SGN': '#4dc98b', 'E18.5 | Undifferentiated SGN': '#55bee0'}
E17_CURATED_LABELS = {}
E9_LOWER_CONFIDENCE_IDS = []
_REVIEWED_METADATA_CONFIGURED = False
EARLY_RAW_LIBRARIES = {'GSM5401072': ('E9.5', 'Data/E9.5_GSE178931/GSM5401072_E9_5_raw_count.csv.gz', 24344, 23777), 'GSM5401073': ('E11.5', 'Data/E11.5_GSE178931/GSM5401073_E11_5_1_raw_count.csv.gz', 8320, 8194), 'GSM5401074': ('E11.5', 'Data/E11.5_GSE178931/GSM5401074_E11_5_2_raw_count.csv.gz', 8553, 8432), 'GSM5401075': ('E11.5', 'Data/E11.5_GSE178931/GSM5401075_E11_5_3_raw_count.csv.gz', 9142, 9036), 'GSM5401076': ('E13.5', 'Data/E13.5_GSE178931/GSM5401076_E13_5_1_raw_count.csv.gz', 8218, 7998), 'GSM5401077': ('E13.5', 'Data/E13.5_GSE178931/GSM5401077_E13_5_2_raw_count.csv.gz', 8568, 8367), 'GSM5401078': ('E13.5', 'Data/E13.5_GSE178931/GSM5401078_E13_5_3_raw_count.csv.gz', 7952, 7776), 'GSM5401079': ('E13.5', 'Data/E13.5_GSE178931/GSM5401079_E13_5_4_raw_count.csv.gz', 9302, 9005)}
SOURCE_PLOT_PREFIXES = {'Source_E9_otic106_HVG3000': 'Source_E9_CVG12_HVG3000', 'Source_E11_otic645_HVG3000': 'Source_E11_CVG_SGN331_HVG3000', 'Source_E13_otic1333_HVG3000': 'Source_E13_CVG_SGN818_HVG3000', 'Source_E15_all740_HVG3000': 'Source_E15_SGN161_sourceContext740_HVG3000', 'Source_E17_all367_HVG3000': 'Source_E17_SGN71_sourceContext367_HVG3000'}

def configure_reviewed_metadata(e17_curated_labels, e9_lower_confidence_ids):
    """Install the reviewed per-cell display evidence supplied by the caller."""
    global E17_CURATED_LABELS, E9_LOWER_CONFIDENCE_IDS, _REVIEWED_METADATA_CONFIGURED
    labels = {str(k): str(v) for k, v in dict(e17_curated_labels).items()}
    low_confidence = list(map(str, e9_lower_confidence_ids))
    assert len(labels) == 71 and len(set(low_confidence)) == len(low_confidence) == 3
    assert all((k.startswith('GSE165502__SS2_20_143__') for k in labels))
    assert set(labels.values()) == {'Type Ia + Type II precursor candidate', 'Type Ib + Type Ic precursor candidate'}
    E17_CURATED_LABELS = labels
    E9_LOWER_CONFIDENCE_IDS = low_confidence
    _REVIEWED_METADATA_CONFIGURED = True

def array_hash(values):
    values = np.ascontiguousarray(values)
    digest = hashlib.sha256()
    digest.update(str(values.shape).encode())
    digest.update(str(values.dtype).encode())
    digest.update(values.tobytes())
    return digest.hexdigest()

def counts_hash(adata, counts_layer='counts'):
    """Hash counts plus ordered cell/gene identities; never modify the source."""
    matrix = sparse.csr_matrix(adata.layers[counts_layer], copy=True)
    matrix.sum_duplicates()
    matrix.sort_indices()
    digest = hashlib.sha256()
    for names in (adata.obs_names, adata.var_names):
        digest.update(json.dumps(list(map(str, names)), ensure_ascii=False).encode())
    for values in (np.asarray(matrix.shape), matrix.indptr.astype(np.int64), matrix.indices.astype(np.int64), matrix.data):
        digest.update(array_hash(values).encode())
    return digest.hexdigest()

def compute_matched_pca(a, ordered_genes, parameters):
    parameters = dict(parameters)
    assert parameters['mask_var'] is None, 'The exact selected genes must all enter PCA.'
    n_pcs = parameters['n_comps']
    scale_clip = parameters['scale_clip']
    genes = pd.Index(ordered_genes).astype(str)
    assert genes.is_unique and n_pcs < min(len(genes), a.n_obs)
    assert genes.isin(a.var_names).all()
    data = a[:, genes].copy()
    unit_variance = parameters.get('unit_variance', True)
    if unit_variance:
        z = data.X.toarray() if sparse.issparse(data.X) else np.asarray(data.X).copy()
        data.X = np.clip((z - z.mean(0)) / np.maximum(z.std(0, ddof=1), 1e-08), -scale_clip, scale_clip)
    else:
        scale_clip = None
    entries = data.X.data if sparse.issparse(data.X) else data.X
    assert np.isfinite(entries).all()
    sc.tl.pca(data, n_comps=n_pcs, svd_solver=parameters['svd_solver'], random_state=parameters['random_state'], mask_var=parameters['mask_var'])
    assert data.varm['PCs'].shape == (len(genes), n_pcs)
    assert data.uns['pca']['params']['mask_var'] is None
    audit = dict(n_input_genes=len(genes), n_pca_genes=len(genes), pca_gene_ids=genes.tolist(), mask_var=None, n_comps=n_pcs, svd_solver=parameters['svd_solver'], random_state=parameters['random_state'], zero_center=True, unit_variance=unit_variance, scale_clip=scale_clip, pca_loadings_sha256=array_hash(data.varm['PCs']))
    return (np.asarray(data.obsm['X_pca'], dtype=np.float32).copy(), audit)

def build_final_graph(obs, pc, corrected, parameters, *, n_threads):
    parameters = dict(parameters)
    b = ad.AnnData(obs=obs.copy())
    b.obsm['X_matched_pca'] = np.asarray(pc, dtype=np.float32).copy()
    b.obsm['X_rep'] = np.asarray(corrected, dtype=np.float32).copy()
    with threadpool_limits(limits=n_threads):
        sc.pp.neighbors(b, n_neighbors=parameters['n_neighbors'], use_rep='X_rep', metric=parameters['metric'], transformer=parameters['transformer'], random_state=parameters['random_state'])
        sc.tl.leiden(b, resolution=parameters['resolution'], flavor=parameters['flavor'], directed=parameters['directed'], n_iterations=parameters['n_iterations'], random_state=parameters['random_state'])
        sc.tl.umap(b, min_dist=parameters['min_dist'], spread=parameters['spread'], random_state=parameters['random_state'])
    b.uns['downstream_parameters_json'] = json.dumps(parameters)
    return b

def curated_type_name(stage, original, cell_id=None):
    if original == 'Outside final target scope':
        return original
    if stage == 'E17.5' and cell_id in E17_CURATED_LABELS:
        return E17_CURATED_LABELS[cell_id]
    if stage == 'E9.5':
        return 'CVG neuroblast'
    if stage in ('E11.5', 'E13.5') and original == 'Undifferentiated SGN':
        return 'Differentiating SGN'
    if stage == 'E13.5' and original == 'CVG':
        return 'Late CVG'
    if stage == 'E16.5' and original == 'Immature Type Ic SGN':
        return 'Emerging Type Ic'
    if stage == 'E17.5' and original == 'Immature Type Ic SGN':
        return 'Type Ib + Type Ic precursor candidate'
    return original

def apply_curated_display(obs):
    """Add reviewed display fields without changing original or training labels."""
    if obs['stage_standardized'].astype(str).isin(['E9.5', 'E17.5']).any():
        assert _REVIEWED_METADATA_CONFIGURED, 'Call configure_reviewed_metadata before plotting E9.5 or E17.5'
    out = obs.copy()
    before = out['cell_type'].astype(str).copy()
    stages = out['stage_standardized'].astype(str)
    out['curated_cell_type'] = [curated_type_name(s, t, str(c)) for c, s, t in zip(out.index, stages, before)]
    out['cell_type_plot'] = out['curated_cell_type'].astype(str)
    out['stage_celltype'] = stages + ' | ' + out['cell_type_plot']
    out['E9_marker_support'] = 'Not applicable'
    e9 = (stages == 'E9.5') & (out['curated_cell_type'] == 'CVG neuroblast')
    out.loc[e9, 'E9_marker_support'] = 'Neurod1 and Insm1 positive (9-cell primary support)'
    out.loc[e9 & out.index.isin(E9_LOWER_CONFIDENCE_IDS), 'E9_marker_support'] = 'Low confidence: weak Neurod1, Insm1 undetected (3 retained cells)'
    assert out['cell_type'].astype(str).equals(before)
    return out

def display_category(key, wrapped=False):
    stage, state = key.split(' | ', 1)
    name = DISPLAY_TYPES.get(state, state).replace('_', ' ')
    return stage + '\n' + textwrap.fill(name, 24) if wrapped else stage + ' ' + name

def umap_pair(b, title, palette, category_order, out_dir, prefix, dpi=600):
    POINT_SIZE, POINT_ALPHA, LABEL_FONTSIZE = (UMAP_STYLE[k] for k in ('point_size', 'point_alpha', 'label_fontsize'))
    xy = np.asarray(b.obsm['X_umap'])
    keys = b.obs.stage_celltype.astype(str).to_numpy()
    rng = np.random.default_rng(0)
    order = rng.permutation(len(xy))
    lo = xy.min(0)
    hi = xy.max(0)
    pad = np.maximum((hi - lo) * 0.055, 0.3)
    limits = [(lo[i] - pad[i], hi[i] + pad[i]) for i in range(2)]
    result = {}
    for mode in ('legend', 'labels'):
        fig = plt.figure(figsize=UMAP_STYLE['legend_figsize'] if mode == 'legend' else UMAP_STYLE['labels_figsize'], facecolor='white')
        ax = fig.add_axes([0.09, 0.12, 0.54, 0.78] if mode == 'legend' else [0.13, 0.12, 0.79, 0.79])
        ax.scatter(xy[order, 0], xy[order, 1], c=[palette[g] for g in keys[order]], s=POINT_SIZE, alpha=POINT_ALPHA, linewidths=0)
        ax.set(xlim=limits[0], ylim=limits[1], xlabel='UMAP1', ylabel='UMAP2', title=title)
        ax.set_box_aspect(1)
        ax.tick_params(labelsize=8)
        ax.title.set_fontsize(10)
        for spine in ('right', 'top'):
            ax.spines[spine].set_visible(False)
        for spine in ('left', 'bottom'):
            ax.spines[spine].set_linewidth(1.15)
        if mode == 'legend':
            handles = [Line2D([], [], marker='o', color=palette[g], ls='', markersize=5.2, label=f'{display_category(g)} ({np.sum(keys == g)})') for g in category_order]
            legend = ax.legend(handles=handles, loc='center left', bbox_to_anchor=(1.025, 0.5), fontsize=UMAP_STYLE['legend_fontsize'], frameon=False, handletextpad=0.35, labelspacing=0.55)
            assert len(legend.texts) == len(category_order)
        else:
            labels = []
            for key in category_order:
                points = xy[keys == key]
                center = np.median(points, axis=0)
                anchor = points[np.argmin(np.sum((points - center) ** 2, axis=1))]
                text = ax.text(*anchor, display_category(key, True), ha='center', va='center', fontsize=LABEL_FONTSIZE, color='black', linespacing=0.96, zorder=5, path_effects=[pe.withStroke(linewidth=UMAP_STYLE['label_stroke'], foreground='white')])
                labels.append((text, anchor.copy()))
            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            bounds = ax.get_window_extent(renderer)
            accepted = []
            placement_order = sorted(range(len(labels)), key=lambda i: -np.sum(keys == category_order[i]))
            offsets = [np.array([0.0, 0.0])] + [radius * np.array([np.cos(angle), np.sin(angle)]) for radius in (12, 22, 35, 50, 70, 95, 125) for angle in np.arange(0, 2 * np.pi, np.pi / 8)]
            for i in placement_order:
                t, anchor = labels[i]
                center = ax.transData.transform(anchor)
                original = t.get_window_extent(renderer)
                candidates = []
                for offset in offsets:
                    box = original.translated(*offset).expanded(1.045, 1.08)
                    outside = max(bounds.x0 + 2 - box.x0, 0) + max(box.x1 - bounds.x1 + 2, 0) + max(bounds.y0 + 2 - box.y0, 0) + max(box.y1 - bounds.y1 + 2, 0)
                    overlaps = sum((box.overlaps(other) for other in accepted))
                    candidates.append((100000.0 * overlaps + 10000.0 * outside + np.linalg.norm(offset), offset, box))
                score, offset, box = min(candidates, key=lambda value: value[0])
                t.set_position(ax.transData.inverted().transform(center + offset))
                accepted.append(box)
            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            actual = [t.get_window_extent(renderer).expanded(1.01, 1.01) for t, _ in labels]
            collisions = sum((actual[i].overlaps(actual[j]) for i in range(len(actual)) for j in range(i)))
            if collisions:
                raise RuntimeError(f'{collisions} overlapping UMAP labels require local position overrides')
            for t, anchor in labels:
                box = t.get_window_extent(renderer)
                ap = ax.transData.transform(anchor)
                distance = np.linalg.norm(ap - np.clip(ap, [box.x0, box.y0], [box.x1, box.y1]))
                if distance > UMAP_STYLE['connector_distance']:
                    target = np.clip(ap, [box.x0, box.y0], [box.x1, box.y1])
                    p = ax.transData.inverted().transform(target)
                    ax.plot([anchor[0], p[0]], [anchor[1], p[1]], color='0.32', lw=UMAP_STYLE['connector_width'], alpha=UMAP_STYLE['connector_alpha'], zorder=4)
            assert len(labels) == len(category_order)
        fig.canvas.draw()
        ext = ax.get_window_extent()
        assert abs(ext.width - ext.height) < 0.01
        # Set the figure output file or directory path.
        path = ROOT
        fig.savefig(path, dpi=dpi, facecolor='white')
        plt.close(fig)
        result[mode] = str(path)
    return result

def stage_cluster_statistics(b, panel):
    """Fixed stage x reviewed identity; measured-only marker denominators."""
    obs = apply_curated_display(b.obs)
    assert np.array_equal(obs.index.astype(str), panel['cell_ids'].astype(str))
    keys = obs['stage_celltype'].astype(str).to_numpy()
    order = [key for key in CURATED_CATEGORY_ORDER if key in set(keys)]
    assert set(order) == set(keys), 'Unreviewed identity must not enter the main plot'
    raw = np.asarray(panel['raw_counts'])
    log = np.asarray(panel['log_cp10k'])
    measured = np.asarray(panel['measured'], dtype=bool)
    assert raw.shape == log.shape == measured.shape
    assert np.isnan(log[~measured]).all()
    avg = []
    pct = []
    coverage = []
    counts = []
    names = []
    for key in order:
        mask = keys == key
        num = measured[mask].sum(0)
        n = int(mask.sum())
        avg.append(np.divide(np.where(measured[mask], log[mask], 0).sum(0), num, out=np.full(raw.shape[1], np.nan), where=num > 0))
        pct.append(np.divide(((raw[mask] > 0) & measured[mask]).sum(0) * 100, num, out=np.full(raw.shape[1], np.nan), where=num > 0))
        coverage.append(num / n)
        counts.append(n)
        names.append(textwrap.fill(display_category(key), 34) + f' (n={n})')
    avg = np.asarray(avg)
    pct = np.asarray(pct)
    coverage = np.asarray(coverage)
    scaled = np.full_like(avg, np.nan)
    for j in range(avg.shape[1]):
        valid = np.isfinite(avg[:, j])
        v = avg[valid, j]
        if len(v):
            scaled[valid, j] = (v - v.mean()) / v.std(ddof=0) if v.std(ddof=0) > 0 else 0
    mode = 'mean_log_cp10k' if len(order) == 1 else 'gene_z_score'
    values = avg if mode == 'mean_log_cp10k' else scaled
    vmax = max(float(np.nanmax(avg)), 0.01) if mode == 'mean_log_cp10k' else 1.0
    assert np.array_equal(np.isnan(avg), coverage == 0)
    assert np.nanmin(pct) >= 0 and np.nanmax(pct) <= 100 and (sum(counts) == len(obs))
    return dict(row_names=names, row_keys=order, n_cells=counts, mean=avg, pct=pct, scaled=scaled, coverage=coverage, color_values=values, color_mode=mode, color_min=0.0 if mode == 'mean_log_cp10k' else -1.0, color_max=vmax)

def dotplot_figure(stats, panel, indices, title, out_path, dpi=600):
    MAX_DOT_AREA, COLUMN_GAP_RATIO, GENE_FONT, ROW_FONT = (DOTPLOT_STYLE[k] for k in ('max_dot_area', 'column_gap_ratio', 'gene_font', 'row_font'))
    names = panel['gene_symbols'][indices].astype(object).copy()
    partial = np.any((stats['coverage'][:, indices] > 0) & (stats['coverage'][:, indices] < 1), axis=0)
    names = np.array([str(n) + '†' if p else str(n) for n, p in zip(names, partial)], dtype=object)
    groups = panel['marker_groups'][indices]
    rows = stats['row_names']
    nr, nc = (len(rows), len(names))
    pitch = (1 + COLUMN_GAP_RATIO) * np.sqrt(MAX_DOT_AREA) / 72
    matrix_width = nc * pitch
    matrix_height = nr * max(DOTPLOT_STYLE['row_pitch'], 0.12 * max((name.count('\n') + 1 for name in rows)) + 0.09)
    left, right, bottom = (3.2, 0.3, 1.1)
    top = 2.65 + max(0, 0.045 * matrix_height - 0.3)
    fw = max(left + matrix_width + right, 5.4)
    fh = bottom + matrix_height + top
    fig = plt.figure(figsize=(fw, fh), facecolor='white')
    ax = fig.add_axes([left / fw, bottom / fh, matrix_width / fw, matrix_height / fh])
    yy, xx = np.indices((nr, nc))
    size = stats['pct'][:, indices] * MAX_DOT_AREA / 100
    values = stats['color_values'][:, indices]
    valid = np.isfinite(size) & (size > 0)
    cmap = LinearSegmentedColormap.from_list('expression_purple', ['#F1F1F1', '#E8C6DE', '#C67ABA', '#952E7D', '#69005F'])
    norm = Normalize(stats['color_min'], stats['color_max'], clip=True)
    ax.scatter(xx[valid], yy[valid], s=size[valid], c=values[valid], cmap=cmap, norm=norm, linewidths=0)
    missing = stats['coverage'][:, indices] == 0
    ax.scatter(xx[missing], yy[missing], s=6, marker='x', color='.77', linewidths=0.45)
    ax.set(xlim=(-0.5, nc - 0.5), ylim=(nr - 0.5, -0.5))
    ax.set_xticks(range(nc), names, rotation=DOTPLOT_STYLE['gene_rotation'], ha='right', fontsize=GENE_FONT)
    ax.set_yticks(range(nr), rows, fontsize=ROW_FONT)
    ax.tick_params(axis='both', length=0, pad=3)
    ax.set_frame_on(False)
    allgroups = list(dict.fromkeys(panel['marker_groups']))
    pool = ['#8A5B78', '#4B7E88', '#A7773E', '#647B4C', '#6874A2', '#9B6461', '#4B8982', '#8C7954', '#8C6290', '#82743D', '#56769D', '#9A6F88', '#7C8E53', '#956444', '#59698E', '#7B6A91', '#66826D', '#9D7658', '#887393', '#65878D', '#98706A', '#828453', '#798D9A', '#9A775C']
    colors = {g: pool[i] for i, g in enumerate(allgroups)}
    spans = []
    for group in dict.fromkeys(groups):
        pos = np.flatnonzero(groups == group)
        a, b = (pos[0], pos[-1])
        spans.append((group, a, b))
        ax.plot([(a + 0.1) / nc, (b + 0.9) / nc], [1.025, 1.025], transform=ax.transAxes, color=colors[group], lw=2, clip_on=False)
        short = GROUP_NAMES.get(group, group)
        ax.text((a + b + 1) / (2 * nc), 1.045, short, rotation=50, ha='center', va='bottom', fontsize=6.8, transform=ax.transAxes)
    legend_y = (bottom + matrix_height + top - 0.9) / fh
    legend_left = left if fw >= left + 5.0 + 0.3 else max(0.3, (fw - 4.8) / 2)
    colorax = fig.add_axes([legend_left / fw, legend_y, 2.15 / fw, 0.1 / fh])
    bar = matplotlib.colorbar.ColorbarBase(colorax, cmap=cmap, norm=norm, orientation='horizontal', ticks=np.linspace(stats['color_min'], stats['color_max'], 5))
    bar.outline.set_visible(False)
    bar.ax.tick_params(labelsize=6, length=2, pad=1)
    bar.ax.set_title('Mean log1p(CP10k)' if stats['color_mode'] == 'mean_log_cp10k' else 'Avg. Exp. (gene-wise z score)', fontsize=7, pad=3)
    sizeleft = legend_left + 2.65
    sizewidth = 2.15
    sizeax = fig.add_axes([sizeleft / fw, legend_y, sizewidth / fw, 0.29 / fh])
    sizeax.set(xlim=(-0.6, 4.6), ylim=(-0.7, 0.7))
    sizeax.axis('off')
    for i, p in enumerate([0, 25, 50, 75, 100]):
        if p:
            sizeax.scatter(i, 0.05, s=p * MAX_DOT_AREA / 100, color='#952E7D', linewidths=0)
        sizeax.text(i, -0.48, str(p), ha='center', fontsize=6)
    sizeax.text(2, 0.6, '% Exp.', ha='center', fontsize=7)
    title_width = max(24, int((fw - left - 0.3) * 72 / (9 * 0.5)))
    title_artist = fig.text(left / fw, 1 - 0.18 / fh, textwrap.fill(title, title_width), ha='left', va='top', fontsize=9)
    fig.canvas.draw()
    title_renderer = fig.canvas.get_renderer()
    right_limit = fig.bbox.x1 - 0.3 * fig.dpi
    while title_artist.get_window_extent(title_renderer).x1 > right_limit and title_width > 12:
        title_width -= 1
        title_artist.set_text(textwrap.fill(title, title_width))
        fig.canvas.draw()
        title_renderer = fig.canvas.get_renderer()
    title_box = title_artist.get_window_extent(title_renderer)
    assert title_box.x1 <= right_limit + 0.01, 'Figure title exceeds its measured right margin'
    legend_top = max(colorax.title.get_window_extent(title_renderer).y1, max((t.get_window_extent(title_renderer).y1 for t in sizeax.texts)))
    assert title_box.y0 >= legend_top + 3, 'Figure title overlaps a top legend; shorten the title'
    fig.text(0.25 / fw, 0.1 / fh, 'x: unmeasured; blank: measured zero. †: partially measured (including Cpne9); statistics use measured cells only.\nRows: fixed stage x reviewed identity, matching UMAP counts. Fixed-annotation DotPlots do not assess integration quality.\nSingle-identity panels show mean log1p(CP10k); all split panels retain the full-panel color scale.', ha='left', va='bottom', fontsize=5.7, color='.30')
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    column_px = ax.get_window_extent(r).width / nc
    diameter_px = np.sqrt(MAX_DOT_AREA) * fig.dpi / 72
    assert abs(column_px / diameter_px - (1 + COLUMN_GAP_RATIO)) < 0.0001
    assert np.allclose(size, stats['pct'][:, indices] * MAX_DOT_AREA / 100, equal_nan=True)
    fig.savefig(out_path, dpi=dpi, facecolor='white')
    plt.close(fig)

def dotplot_set(b, panel, title, out_dir, prefix, dpi=600):
    title = 'Fixed reviewed identities | measured log1p(CP10k)' if b.n_obs == 7355 else title
    stats = stage_cluster_statistics(b, panel)
    n = len(panel['gene_symbols'])
    paths = []
    # Set the figure output file or directory path.
    path = ROOT
    dotplot_figure(stats, panel, np.arange(n), title, path, dpi)
    paths.append(str(path))
    blocks = []
    current = []
    for group in dict.fromkeys(panel['marker_groups']):
        ix = np.flatnonzero(panel['marker_groups'] == group).tolist()
        if len(current) + len(ix) > DOTPLOT_STYLE['split_max_genes'] and current:
            blocks.append(current)
            current = []
        current.extend(ix)
    if current:
        blocks.append(current)
    for i, ix in enumerate(blocks, 1):
        # Set the figure output file or directory path.
        path = ROOT
        dotplot_figure(stats, panel, np.array(ix), title, path, dpi)
        paths.append(str(path))
    return (paths, stats)

def subset_measured_panel(panel, keep, expected_ids):
    """Subset every cell-axis field, never a gene-axis field."""
    keep = np.asarray(keep, dtype=bool)
    result = dict(panel)
    for key in ('cell_ids', 'raw_counts', 'log_cp10k', 'measured', 'common_total_counts'):
        assert np.asarray(panel[key]).shape[0] == len(keep)
        result[key] = np.asarray(panel[key])[keep].copy()
    assert np.array_equal(result['cell_ids'].astype(str), np.asarray(expected_ids).astype(str))
    return result

def early_feature_flags(symbols):
    symbols = pd.Index(np.asarray(symbols, dtype=str))
    low = symbols.str.lower()
    var = pd.DataFrame(index=symbols.rename('feature_row_id_index'))
    for field in ('gene_symbol', 'gene_id_analysis', 'gene_id_original', 'source_feature_id', 'canonical_gene_id'):
        var[field] = symbols.to_numpy()
    var['mt_feature'] = low.str.startswith('mt-')
    var['ribosomal_feature'] = low.str.startswith(('rpl', 'rps')) & ~low.str.startswith('rps6k')
    var['ERCC_feature'] = low.str.startswith('ercc-')
    var['reporter_or_vector_feature'] = low.isin(['egfp', 'gfp', 'tdtomato', 'tomato', 'wpre', 'mcherry'])
    var['technical_feature'] = var.ERCC_feature | var.reporter_or_vector_feature
    var['external_spike_feature'] = False
    var['eligible_for_within_dataset_analysis'] = ~var.technical_feature
    return var

def read_early_csv(root, library, requested_global_ids=None, *, chunk_genes=128, nmads=4.0, max_mito_pct=15.0, verify_historical_qc=True):
    """Return raw-count AnnData on all 31054 genes, without removing QC failures.

    Full-library metrics and thresholds are computed from every original cell.
    requested_global_ids only controls matrix materialization and returned order.
    Mother membership remains a separate frozen author/source review decision.
    """
    stage, relative, expected_raw_n, expected_pass_n = EARLY_RAW_LIBRARIES[library]
    # Read the independently configured raw-count file for the current source library.
    path = EARLY_RAW_COUNT_FILES[library]
    with gzip.open(path, 'rt', encoding='utf-8') as stream:
        header = next(csv.reader(stream))
    assert header[0] == '' and len(header) - 1 == expected_raw_n
    barcodes = pd.Index(header[1:], dtype=str)
    assert barcodes.is_unique and barcodes.str.fullmatch('[ACGT]{16}\\.1').all()
    ids = pd.Index('GSE178931__' + library + '__' + barcodes, name='cell_id_global')
    requested = ids if requested_global_ids is None else pd.Index(requested_global_ids, dtype=str)
    positions = ids.get_indexer(requested)
    assert requested.is_unique and (positions >= 0).all(), 'Unknown or duplicated requested cell'
    metrics = {k: np.zeros(expected_raw_n, dtype=np.int64) for k in ('total_counts_source', 'n_features_source', 'total_counts_endogenous', 'n_genes_endogenous', 'mito_counts')}
    pieces, genes, gene_n, gene_totals = ([], [], [], [])
    for block in pd.read_csv(path, index_col=0, chunksize=chunk_genes, keep_default_na=False, na_filter=False):
        assert np.array_equal(block.columns.to_numpy(dtype=str), barcodes.to_numpy())
        values = block.to_numpy()
        assert np.isfinite(values).all() and (values >= 0).all()
        assert np.equal(values, np.rint(values)).all() and values.max() <= np.iinfo(np.int32).max
        values = values.astype(np.int32)
        flags = early_feature_flags(block.index.astype(str))
        endogenous = ~flags.technical_feature.to_numpy()
        metrics['total_counts_source'] += values.sum(0, dtype=np.int64)
        metrics['n_features_source'] += (values > 0).sum(0)
        metrics['total_counts_endogenous'] += values[endogenous].sum(0, dtype=np.int64)
        metrics['n_genes_endogenous'] += (values[endogenous] > 0).sum(0)
        metrics['mito_counts'] += values[flags.mt_feature.to_numpy()].sum(0, dtype=np.int64)
        genes.extend(block.index.astype(str))
        gene_n.extend((values > 0).sum(1).tolist())
        gene_totals.extend(values.sum(1, dtype=np.int64).tolist())
        pieces.append(sparse.csr_matrix(values[:, positions].T))
    assert len(genes) == len(set(genes)) == 31054
    var = early_feature_flags(genes)
    var['n_cells'] = np.asarray(gene_n, dtype=np.int32)
    var['total_counts'] = np.asarray(gene_totals, dtype=np.int64)
    var['zero_count_gene'] = var.total_counts.eq(0)
    full = pd.DataFrame(metrics, index=ids)
    full['zero_endogenous'] = full.total_counts_endogenous.eq(0) | full.n_genes_endogenous.eq(0)
    full['log10_genes_endogenous'] = np.log10(1.0 + full.n_genes_endogenous)
    valid = ~full.zero_endogenous
    values = full.loc[valid, 'log10_genes_endogenous'].to_numpy()
    center = float(np.median(values))
    spread = float(median_abs_deviation(values, scale='normal'))
    assert len(values) >= 30 and np.isfinite(spread) and (spread > 1e-12)
    lo, hi = (max(0.0, center - nmads * spread), max(0.0, center + nmads * spread))
    full['pct_mito'] = np.divide(100.0 * full.mito_counts, full.total_counts_endogenous, out=np.full(expected_raw_n, np.nan), where=full.total_counts_endogenous.to_numpy() > 0)
    full['fail_zero_expression'] = full.zero_endogenous
    full['fail_low_complexity'] = valid & full.log10_genes_endogenous.lt(lo)
    full['fail_high_complexity'] = valid & full.log10_genes_endogenous.gt(hi)
    full['fail_high_mito'] = full.pct_mito.gt(max_mito_pct)
    full['fail_missing_qc_metric'] = ~np.isfinite(full.pct_mito)
    full['qc_pass'] = ~full.filter(regex='^fail_').any(axis=1)
    full['doublet_review'] = full.fail_high_complexity
    if verify_historical_qc:
        assert int(full.qc_pass.sum()) == expected_pass_n, (library, int(full.qc_pass.sum()), expected_pass_n)
    obs = full.loc[requested].copy()
    obs['original_cell_id'] = barcodes[positions].to_numpy()
    obs['source_cell_id'] = obs.original_cell_id
    obs['cell_id_global'] = requested.to_numpy()
    obs['library_id'] = library
    obs['GSM'] = library
    obs['study_id'] = obs['dataset_id'] = 'GSE178931'
    for field in ('stage_standardized', 'stage_resolved', 'stage_original', 'stage_analysis'):
        obs[field] = stage
    obs['source_file'] = relative
    obs['technology'] = '10x Chromium'
    obs['count_unit_verified'] = 'UMI'
    obs['source_object_key'] = library
    counts = sparse.hstack(pieces, format='csr', dtype=np.int32)
    counts.sort_indices()
    result = ad.AnnData(counts, obs=obs, var=var)
    result.uns['raw_source_qc'] = dict(source_path=relative, raw_cells=expected_raw_n, pass_cells=int(full.qc_pass.sum()), returned_cells=len(obs), source_genes=31054, metrics_scope='all original library cells before requested-ID selection', rule='nonzero endogenous; log10(1+genes) +/-4 normal-scaled MAD; mt<=15%', nmads=float(nmads), max_mito_pct=float(max_mito_pct), median_log10_genes=center, scaled_mad_log10_genes=spread, low_log10_genes=lo, high_log10_genes=hi, identity_filter='none; reviewed mother/retained membership supplied separately')
    return result

def _source_counts(raw):
    counts = sparse.csr_matrix(raw.layers.get('counts', raw.X), copy=True)
    counts.sum_duplicates()
    counts.sort_indices()
    assert np.isfinite(counts.data).all() and (counts.data >= 0).all()
    assert np.equal(counts.data, np.rint(counts.data)).all()
    return counts

def _source_symbols_source(raw):
    field = next((key for key in ('gene_symbol', 'gene_id_analysis') if key in raw.var), None)
    return pd.Index(raw.var[field].astype(str) if field else raw.var_names.astype(str))

def _source_counts_hash(raw, counts):
    digest = hashlib.sha256()
    for names in (raw.obs_names, raw.var_names):
        digest.update(json.dumps(list(map(str, names)), ensure_ascii=False).encode())
    for values in (np.asarray(counts.shape), counts.indptr.astype(np.int64), counts.indices.astype(np.int64), counts.data):
        values = np.ascontiguousarray(values)
        part = hashlib.sha256()
        part.update(str(values.shape).encode())
        part.update(str(values.dtype).encode())
        part.update(values.tobytes())
        digest.update(part.hexdigest().encode())
    return digest.hexdigest()

def _source_symbol_projection(counts, symbols, requested):
    """Sum native rows sharing a requested symbol; retain measurement separately."""
    requested = pd.Index(requested, dtype=str)
    assert requested.is_unique
    if symbols.is_unique:
        positions = symbols.get_indexer(requested)
        present = positions >= 0
        if present.all():
            return (counts[:, positions].copy(), present)
    else:
        present = requested.isin(symbols)
    destinations = requested.get_indexer(symbols)
    rows = np.flatnonzero(destinations >= 0)
    mapping = sparse.csr_matrix((np.ones(len(rows), dtype=np.int64), (rows, destinations[rows])), shape=(len(symbols), len(requested)))
    result = counts.astype(np.int64) @ mapping
    return (result.tocsr(), np.asarray(present, dtype=bool))

def attach_source_review(raw, case, approvedobs):
    """Select the frozen mother pool and attach display metadata on a copy."""
    ids = pd.Index(case['cells'], dtype=str)
    assert ids.is_unique and raw.obs_names.is_unique and ids.isin(raw.obs_names).all()
    assert len(ids) == case['n_cells'] and approvedobs.index.is_unique
    result = raw[ids].copy()
    result.layers['counts'] = _source_counts(result)
    obs = result.obs.copy()
    obs['original_annotation_source'] = obs.get('cell_type', pd.Series('No source subtype annotation', index=ids)).astype(str)
    retained = ids.isin(case['reviewed_labels'])
    retained_ids = ids[retained]
    assert retained_ids.isin(approvedobs.index).all()
    approved = approvedobs.loc[retained_ids]
    for field, expected in [('stage_standardized', case['stage']), ('study_id', case['study'])]:
        if field in approved:
            assert approved[field].astype(str).eq(expected).all(), (case['tag'], field)
        if field in obs:
            assert obs[field].dropna().astype(str).eq(expected).all(), (case['tag'], field)
        obs[field] = expected
    if 'library_id' in approved and 'library_id' in obs:
        assert obs.loc[retained_ids, 'library_id'].astype(str).equals(approved.library_id.astype(str))
    for field in approved.columns:
        if field not in obs:
            obs[field] = approved[field].reindex(ids)
    for field in ('integration_batch', 'biological_replicate_id', 'cochlear_region'):
        if field in approved:
            if field in obs:
                values = obs[field].astype(object).copy()
            else:
                values = pd.Series(pd.NA, index=ids, dtype=object)
            values.loc[retained_ids] = approved[field].astype(object).to_numpy()
            obs[field] = values
    if 'integration_batch' not in obs:
        obs['integration_batch'] = obs['library_id'].astype(object)
    else:
        obs['integration_batch'] = obs['integration_batch'].astype(object).where(obs['integration_batch'].notna(), obs['library_id'].astype(object))
    obs['source_reviewed_cell_type'] = [case['reviewed_labels'].get(c, 'Outside final target scope') for c in ids]
    obs['cell_type'] = obs['source_reviewed_cell_type'].astype(object)
    if 'cell_type' in approved:
        obs.loc[retained_ids, 'cell_type'] = approved.cell_type.astype(object).to_numpy()
    obs['retained_in_atlas'] = retained
    obs['source_rna_mixture_review_flag'] = ids.isin(case.get('mixed_rna_cells', []))
    obs['stage_celltype'] = case['stage'] + ' | ' + obs.cell_type.astype(str)
    result.obs = obs
    result.uns['source_case'] = case['tag']
    assert result.obs_names.equals(ids)
    return result

def prepare_source_expression(raw, case, common_var, groups133, target_sum=10000):
    """Normalize once on the explicit case axis; extract markers from native rows.

    E9 uses source_eligible_axis. E14/E16 UMI require the explicit
    normalization_exclude_hemoglobin flag to reproduce their 17991-gene axis.
    Other source-endogenous cases retain hemoglobin in their denominator.
    common11467 cases sum any duplicated native symbols before normalization.
    """
    counts = _source_counts(raw)
    native_hash = _source_counts_hash(raw, counts)
    symbols = _source_symbols_source(raw)
    assert target_sum > 0 and raw.obs_names.is_unique
    normalization = case['normalization']
    if normalization == 'common11467':
        ordered = pd.Index(common_var['gene_symbol'].astype(str))
        assert common_var.index.is_unique and ordered.is_unique and (len(ordered) == 11467)
        selected, present = _source_symbol_projection(counts, symbols, ordered)
        assert present.all(), ordered[~present].tolist()
        excluded_symbols = symbols[~symbols.isin(ordered)].tolist()
    else:
        assert normalization == 'source_endogenous' and symbols.is_unique
        excluded = np.zeros(raw.n_vars, dtype=bool)
        for key in ('mt_feature', 'technical_feature', 'ERCC_feature', 'external_spike_feature', 'reporter_or_vector_feature'):
            if key in raw.var:
                excluded |= raw.var[key].fillna(False).to_numpy(dtype=bool)
        excluded |= symbols.str.lower().str.startswith('mt-')
        if case.get('source_eligible_axis', False):
            assert 'eligible_for_within_dataset_analysis' in raw.var
            excluded |= ~raw.var.eligible_for_within_dataset_analysis.fillna(False).to_numpy(dtype=bool)
            excluded |= symbols.str.lower().str.startswith(('hba', 'hbb'))
        if case.get('normalization_exclude_hemoglobin', False):
            excluded |= symbols.str.match('(?i)^Hb[ab]-')
        positions = np.flatnonzero(~excluded)
        ordered = symbols[positions]
        if case.get('normalization_gene_symbols'):
            frozen_axis = pd.Index(case['normalization_gene_symbols'], dtype=str)
            assert frozen_axis.is_unique and set(frozen_axis) == set(ordered)
            positions = symbols.get_indexer(frozen_axis)
            ordered = frozen_axis
        selected = counts[:, positions].copy()
        excluded_symbols = symbols[excluded].tolist()
    if case.get('expected_normalization_genes') is not None:
        assert len(ordered) == case['expected_normalization_genes']
    expression = ad.AnnData(selected.astype(np.float32), obs=raw.obs.copy(), var=pd.DataFrame(index=ordered))
    expression.layers['counts'] = selected.copy()
    denominator = np.asarray(selected.sum(1)).ravel().astype(float)
    assert (denominator > 0).all()
    sc.pp.normalize_total(expression, target_sum=target_sum)
    sc.pp.log1p(expression)
    genes = [g for values in groups133.values() for g in values]
    groups = [group for group, values in groups133.items() for _ in values]
    assert len(genes) == len(set(genes)) == 133
    marker_counts, present = _source_symbol_projection(counts, symbols, genes)
    measured = np.broadcast_to(present, (raw.n_obs, len(genes))).copy()
    marker_raw = marker_counts.toarray().astype(float)
    marker_raw[~measured] = np.nan
    marker_log = np.log1p(marker_raw * target_sum / denominator[:, None])
    assert np.array_equal(~measured, np.isnan(marker_raw))
    assert np.array_equal(~measured, np.isnan(marker_log))
    panel = dict(cell_ids=np.asarray(raw.obs_names, dtype=str), gene_symbols=np.asarray(genes, dtype=str), marker_groups=np.asarray(groups, dtype=str), raw_counts=marker_raw, log_cp10k=marker_log, measured=measured, common_total_counts=denominator)
    audit = dict(source_case=case['tag'], source_native_counts_sha256=native_hash, native_axis_genes=raw.n_vars, normalization=normalization, normalization_axis_genes=expression.n_vars, normalization_target_sum=target_sum, normalization_excluded_symbols=excluded_symbols, normalization_gene_symbols=ordered.tolist(), native_duplicate_symbol_rows=int(symbols.duplicated().sum()), normalization_duplicate_symbol_rule='Sum native rows sharing a symbol', marker_measurement_source='Native source rows before denominator exclusions', measured_marker_genes=int(present.sum()), missing_marker_symbols=np.asarray(genes)[~present].tolist())
    expression.uns['source_preparation'] = audit
    assert _source_counts_hash(raw, _source_counts(raw)) == native_hash
    return (expression, panel, audit)

def fit_source_case(expression, case, hvg, n_threads=4, *, pca_parameters, graph_parameters, hvg_parameters, harmony_parameters, harmony_threads=1):
    """Fit the exact source branch without using reviewed identity labels."""
    before = counts_hash(expression)
    if case.get('exact_gene_symbols'):
        genes = pd.Index(case['exact_gene_symbols'][:hvg])
        assert genes.is_unique and genes.isin(expression.var_names).all()
    else:
        eligible = ~expression.var_names.str.match(hvg_parameters['excluded_symbol_regex'])
        excluded_symbols = hvg_parameters.get('excluded_symbols', ())
        if case.get('author1371', False):
            assert len(excluded_symbols) == len(set(excluded_symbols)) == 1371
            eligible &= ~expression.var_names.isin(excluded_symbols)
        base = expression[:, eligible].copy()
        if case.get('min_gene_cells'):
            detected = np.asarray((base.layers['counts'] > 0).sum(0)).ravel()
            base = base[:, detected >= case['min_gene_cells']].copy()
        with threadpool_limits(limits=n_threads):
            flavor = case.get('hvg_flavor') or hvg_parameters['flavor']
            if flavor == 'seurat_v3':
                sc.pp.highly_variable_genes(base, n_top_genes=hvg, flavor='seurat_v3', layer='counts', span=hvg_parameters['seurat_v3_span'])
                genes = base.var_names[base.var.highly_variable]
            else:
                sc.pp.highly_variable_genes(base, n_top_genes=max(hvg_parameters['rank_at_least'], hvg), flavor=flavor)
                ranking = base.var.assign(gene=base.var_names).sort_values(['dispersions_norm', 'gene'], ascending=[False, True], kind='stable')
                genes = ranking.index[:hvg]
    assert len(genes) == hvg and (not genes.str.lower().str.startswith('mt-').any())
    pc_parameters = dict(pca_parameters, n_comps=case['n_pcs'])
    with threadpool_limits(limits=n_threads):
        pc, pca_audit = compute_matched_pca(expression, genes, pc_parameters)
    history = {}
    corrected = pc.copy()
    parameters = dict(graph_parameters, resolution=case['resolution'])
    graph = build_final_graph(expression.obs, pc, corrected, parameters, n_threads=n_threads)
    assert counts_hash(expression) == before
    audit = dict(expression.uns['source_preparation'])
    audit.update(pca_audit)
    audit.update(source_counts_sha256=before, source_expression_counts_sha256=before, n_cells=expression.n_obs, n_clusters=graph.obs.leiden.nunique(), graph_parameters=parameters, hvg_parameters=dict(hvg_parameters), source_labels_used_for_fit=False, source_case=case['tag'], harmony=history)
    graph.uns['source_fit_audit_json'] = json.dumps(audit, default=lambda x: x.item())
    return (graph, audit)

def render_source_results(results, panels, audits, figure_directory, dpi):
    """Render retained cells on untouched source coordinates; return paths/stats.

    The caller sets methods.UMAP_STYLE and methods.DOTPLOT_STYLE explicitly.
    Each source case has its own directory, preserving identical stage basenames
    without colliding across technologies, HVGs, or context/retained-only fits.
    """
    paths, statistics = ({}, {})
    for tag, graph in results.items():
        keep = graph.obs.retained_in_atlas.to_numpy(dtype=bool)
        assert keep.any()
        shown = graph[keep].copy()
        original_coordinates = graph.obsm['X_umap'][keep].copy()
        shown.obs = apply_curated_display(shown.obs)
        panel = subset_measured_panel(panels[tag], keep, shown.obs_names)
        categories = [key for key in CURATED_CATEGORY_ORDER if key in set(shown.obs.stage_celltype)]
        assert set(categories) == set(shown.obs.stage_celltype)
        study = ', '.join(sorted(set(shown.obs.study_id.astype(str))))
        stage = ', '.join(sorted(set(shown.obs.stage_standardized.astype(str))))
        assert shown.obs.stage_standardized.nunique() == 1
        title = f'{stage} | {study} | source-context n={graph.n_obs}\nAtlas-retained n={shown.n_obs}; unchanged source UMAP'
        prefix = SOURCE_PLOT_PREFIXES.get(tag, tag)
        # Set the figure output file or directory path.
        directory = ROOT
        directory.mkdir(parents=True, exist_ok=True)
        umaps = umap_pair(shown, title, CURATED_PALETTE, categories, directory, prefix, dpi=dpi)
        dots, stats = dotplot_set(shown, panel, title.replace('\n', ' | '), directory, prefix, dpi=dpi)
        stage_name = stage.removesuffix('.5')
        old_paths = [umaps['legend'], umaps['labels']] + dots
        names = [f'UMAP_{stage_name}_legend.png', f'UMAP_{stage_name}_label.png', f'Dotplot_{stage_name}_full.png'] + [f'Dotplot_{stage_name}_{i}.png' for i in range(1, len(dots))]
        assert len(old_paths) == len(names)
        renamed = []
        for old, name in zip(old_paths, names):
            # Set the required input, output, or working path for this step.
            destination = ROOT
            Path(old).replace(destination)
            renamed.append(str(destination))
        assert np.array_equal(shown.obsm['X_umap'], original_coordinates)
        assert sum(stats['n_cells']) == int(keep.sum())
        paths[tag], statistics[tag] = (renamed, stats)
        print(f'{prefix}: {graph.n_obs} source-context cells; {shown.n_obs} retained; {len(renamed)} files at {dpi} DPI.')
    return (paths, statistics, dict(CURATED_PALETTE))

def _integer_counts(raw):
    source = raw.layers['counts'] if 'counts' in raw.layers else raw.X
    counts = sparse.csr_matrix(source, copy=True)
    assert counts.shape == raw.shape
    assert np.isfinite(counts.data).all() and (counts.data >= 0).all()
    assert np.equal(counts.data, np.rint(counts.data)).all(), 'Counts must be integers'
    assert counts.data.size == 0 or counts.data.max() <= np.iinfo(np.int32).max
    counts = counts.astype(np.int64)
    counts.sum_duplicates()
    counts.sort_indices()
    counts.eliminate_zeros()
    assert counts.data.size == 0 or counts.data.max() <= np.iinfo(np.int32).max
    return counts

def _source_symbols_storage(raw):
    field = next((key for key in ('gene_symbol', 'gene_id_analysis') if key in raw.var), None)
    values = raw.var[field] if field else pd.Series(raw.var_names, index=raw.var_names)
    return pd.Index(values.astype('string').fillna('').astype(str))

def _symbol_projection(source_symbols, requested):
    requested = pd.Index(requested).astype(str)
    assert requested.is_unique and (np.asarray(requested.str.len()) > 0).all()
    destination = requested.get_indexer(source_symbols)
    keep = np.flatnonzero(destination >= 0)
    projection = sparse.csr_matrix((np.ones(len(keep), dtype=np.int64), (keep, destination[keep])), shape=(len(source_symbols), len(requested)))
    present = np.asarray(projection.sum(axis=0)).ravel() > 0
    return (projection, present)

def project_common(raw, common_var):
    """Sum duplicate source symbols onto the exact measured 11,467-gene axis."""
    assert len(common_var) == 11467 and common_var.index.is_unique
    assert 'gene_symbol' in common_var
    symbols = pd.Index(common_var['gene_symbol'].astype(str))
    assert symbols.is_unique and raw.obs_names.is_unique
    projection, present = _symbol_projection(_source_symbols_storage(raw), symbols)
    assert present.all(), f'Unmeasured common genes: {symbols[~present].tolist()}'
    counts = (_integer_counts(raw) @ projection).tocsr()
    counts.sum_duplicates()
    counts.sort_indices()
    counts.eliminate_zeros()
    assert counts.data.size == 0 or counts.data.max() <= np.iinfo(np.int32).max
    counts = counts.astype(np.int32)
    return ad.AnnData(X=counts.copy(), obs=raw.obs.copy(), var=common_var.copy(), layers={'counts': counts})

def extract_marker_panel(raw, groups133, denominator):
    """Measure 133 source markers; absent features remain NaN, never zero."""
    genes = [str(g) for values in groups133.values() for g in values]
    groups = [str(group) for group, values in groups133.items() for _ in values]
    assert len(genes) == len(set(genes)) == 133
    denominator = np.asarray(denominator, dtype=np.float64).reshape(-1)
    assert len(denominator) == raw.n_obs
    assert np.isfinite(denominator).all() and (denominator > 0).all()
    projection, present = _symbol_projection(_source_symbols_storage(raw), genes)
    summed = (_integer_counts(raw) @ projection).toarray().astype(np.float64)
    measured = np.broadcast_to(present, (raw.n_obs, len(genes))).copy()
    marker_raw = np.where(measured, summed, np.nan)
    log = np.log1p(marker_raw * 10000.0 / denominator[:, None])
    assert np.array_equal(np.isnan(marker_raw), ~measured)
    assert np.array_equal(np.isnan(log), ~measured)
    return dict(raw_counts=marker_raw, log_cp10k=log, measured=measured, cell_ids=np.asarray(raw.obs_names, dtype=str), gene_symbols=np.asarray(genes, dtype=str), marker_groups=np.asarray(groups, dtype=str), common_total_counts=denominator)

def _json_default(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (set, frozenset, tuple)):
        return list(value)
    if value is pd.NA:
        return None
    raise TypeError(f'Unsupported provenance value: {type(value).__name__}')

def provenance_json(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, default=_json_default, allow_nan=False)

def _pack_panel(adata, panel):
    assert np.array_equal(adata.obs_names.astype(str), panel['cell_ids'].astype(str))
    adata.obsm['marker_raw_counts'] = np.asarray(panel['raw_counts'], dtype=np.float64)
    adata.obsm['marker_measured'] = np.asarray(panel['measured'], dtype=bool)
    adata.uns['marker_gene_symbols'] = np.asarray(panel['gene_symbols'], dtype=str)
    adata.uns['marker_groups'] = np.asarray(panel['marker_groups'], dtype=str)
    adata.uns['marker_contract_json'] = provenance_json(dict(n_markers=133, raw_dtype='float64', measured_dtype='bool', target_sum=10000.0, normalization='log1p(marker_raw * 10000. / common11467_total_counts)', missing='NaN means an unmeasured source feature; measured zero remains zero', duplicate_symbols='Source columns with the same gene symbol are summed'))

def panel_from_adata(adata):
    """Restore marker log expression from raw markers and common-axis counts."""
    counts = _integer_counts(adata)
    denominator = np.asarray(counts.sum(axis=1)).ravel().astype(np.float64)
    assert (denominator > 0).all()
    raw = np.asarray(adata.obsm['marker_raw_counts'], dtype=np.float64)
    measured = np.asarray(adata.obsm['marker_measured'], dtype=bool)
    assert raw.shape == measured.shape == (adata.n_obs, 133)
    assert np.array_equal(np.isnan(raw), ~measured)
    return dict(cell_ids=np.asarray(adata.obs_names, dtype=str), gene_symbols=np.asarray(adata.uns['marker_gene_symbols'], dtype=str), marker_groups=np.asarray(adata.uns['marker_groups'], dtype=str), raw_counts=raw.copy(), measured=measured.copy(), log_cp10k=np.log1p(raw * 10000.0 / denominator[:, None]), common_total_counts=denominator)

def make_stage_input(raw_parts, approved_obs_stage, common_var, groups133):
    """Build count-only stage input in approved order; frozen labels never fit data.

    raw_parts is a sequence or mapping of real source AnnData objects. It may
    include mother-context cells, but only approved_obs_stage.index is exported.
    Source-only restored marker features must already be present in raw_parts.
    """
    approved = approved_obs_stage.copy()
    assert len(approved) > 0 and approved.index.is_unique
    assert approved['stage_standardized'].nunique() == 1
    parts = list(raw_parts.values()) if isinstance(raw_parts, Mapping) else list(raw_parts)
    frames, counts_parts, panels = ([], [], [])
    for raw in parts:
        assert raw.obs_names.is_unique
        keep = raw.obs_names.isin(approved.index)
        if not keep.any():
            continue
        source = raw[keep].copy()
        common = project_common(source, common_var)
        denominator = np.asarray(common.layers['counts'].sum(axis=1)).ravel().astype(np.float64)
        panels.append(extract_marker_panel(source, groups133, denominator))
        frames.append(common.obs.copy())
        counts_parts.append(common.layers['counts'])
    assert frames, 'No approved source cells found'
    source_obs = pd.concat(frames, axis=0, sort=False)
    assert source_obs.index.is_unique, 'An approved cell appears in multiple sources'
    assert set(source_obs.index) == set(approved.index), 'Approved membership differs from sources'
    positions = source_obs.index.get_indexer(approved.index)
    source_obs = source_obs.loc[approved.index].copy()
    for key in ('study_id', 'library_id', 'stage_standardized'):
        if key in source_obs and key in approved:
            valid = source_obs[key].notna() & approved[key].notna()
            assert np.array_equal(source_obs.loc[valid, key].astype(str), approved.loc[valid, key].astype(str)), key
    for column in approved:
        source_obs[column] = approved[column].copy()
    counts = sparse.vstack(counts_parts, format='csr', dtype=np.int32)[positions].copy()
    result = ad.AnnData(X=counts.copy(), obs=source_obs, var=common_var.copy(), layers={'counts': counts.copy()})
    panel = dict(panels[0])
    for p in panels[1:]:
        assert np.array_equal(p['gene_symbols'], panel['gene_symbols'])
        assert np.array_equal(p['marker_groups'], panel['marker_groups'])
    for key in ('cell_ids', 'raw_counts', 'log_cp10k', 'measured', 'common_total_counts'):
        panel[key] = np.concatenate([p[key] for p in panels], axis=0)[positions]
    _pack_panel(result, panel)
    result.uns['stage_input_contract_json'] = provenance_json(dict(stage=str(approved['stage_standardized'].iloc[0]), cells=result.n_obs, common_genes=result.n_vars, X='Unnormalized int32 CSR common-axis counts', counts_layer='counts', source_parts=len(frames), embeddings_saved=False, membership='Exactly approved_obs_stage.index in its explicit order', annotation='Source metadata retained; approved metadata overrides reviewed fields'))
    assert result.obs_names.equals(approved.index)
    assert set(result.obsm) == {'marker_raw_counts', 'marker_measured'} and (not result.obsp)
    restored = panel_from_adata(result)
    assert np.array_equal(restored['log_cp10k'], panel['log_cp10k'], equal_nan=True)
    return result

def _h5_frame(frame):
    """Normalize object columns for H5AD while retaining missing values."""
    result = frame.copy()
    for key in result.columns:
        column = result[key]
        if isinstance(column.dtype, pd.CategoricalDtype):
            if column.cat.categories.dtype == object:
                categories = list(column.cat.categories)
                if not (all((isinstance(v, str) for v in categories)) or all((isinstance(v, (int, float, np.number)) for v in categories))):
                    result[key] = pd.Categorical(column.astype('string'), categories=list(map(str, categories)), ordered=column.cat.ordered)
            continue
        if column.dtype == object:
            present = column.dropna()
            if len(present) and present.map(lambda v: isinstance(v, (bool, np.bool_))).all():
                result[key] = column.astype('boolean')
            elif len(present) and present.map(lambda v: isinstance(v, (int, float, np.number))).all():
                result[key] = pd.to_numeric(column)
            else:
                result[key] = column.astype('string')
    result.index = result.index.astype(str)
    return result

def _same_array(left, right):
    if sparse.issparse(left) or sparse.issparse(right):
        left = sparse.csr_matrix(left, copy=True)
        right = sparse.csr_matrix(right, copy=True)
        left.sum_duplicates()
        left.sort_indices()
        right.sum_duplicates()
        right.sort_indices()
        assert left.shape == right.shape and left.dtype == right.dtype
        assert np.array_equal(left.indptr, right.indptr) and np.array_equal(left.indices, right.indices)
        assert np.array_equal(left.data, right.data, equal_nan=True)
    else:
        left, right = (np.asarray(left), np.asarray(right))
        assert left.shape == right.shape and left.dtype == right.dtype
        numerical = left.dtype.kind in 'biufc'
        assert np.array_equal(left, right, equal_nan=numerical)

def save_h5ad_verified(adata, destination, project_root, *, overwrite=False, provenance=None):
    """Stage -> read back -> verify -> atomic replace, with no implicit exports."""
    root = Path(project_root).resolve()
    destination = Path(destination).resolve()
    assert destination.is_relative_to(root), 'Output must remain inside the project'
    assert destination.suffix.lower() == '.h5ad'
    if destination.exists() and (not overwrite):
        raise FileExistsError(destination)
    # Set the temporary working directory or file path.
    staging_dir = ROOT
    staging_dir.mkdir(parents=True, exist_ok=True)
    # Set the temporary working directory or file path.
    staging = ROOT
    prepared = adata.copy()
    prepared.obs = _h5_frame(prepared.obs)
    prepared.var = _h5_frame(prepared.var)
    if provenance is not None:
        prepared.uns['export_provenance_json'] = provenance_json(provenance)
    previous_setting = ad.settings.allow_write_nullable_strings
    try:
        ad.settings.allow_write_nullable_strings = True
        prepared.write_h5ad(staging, compression='gzip', convert_strings_to_categoricals=False)
        checked = ad.read_h5ad(staging)
        assert checked.obs_names.equals(prepared.obs_names) and checked.var_names.equals(prepared.var_names)
        pd.testing.assert_frame_equal(checked.obs, prepared.obs, check_dtype=False, check_categorical=False)
        pd.testing.assert_frame_equal(checked.var, prepared.var, check_dtype=False, check_categorical=False)
        for frame_a, frame_b in ((prepared.obs, checked.obs), (prepared.var, checked.var)):
            for key in frame_a:
                if isinstance(frame_a[key].dtype, pd.CategoricalDtype):
                    assert frame_b[key].cat.categories.equals(frame_a[key].cat.categories)
                    assert frame_b[key].cat.ordered == frame_a[key].cat.ordered
        _same_array(checked.X, prepared.X)
        for slot in ('layers', 'obsm', 'varm', 'obsp', 'varp'):
            expected, actual = (getattr(prepared, slot), getattr(checked, slot))
            assert set(expected) == set(actual)
            for key in expected:
                _same_array(expected[key], actual[key])
        if 'marker_raw_counts' in prepared.obsm:
            before, after = (panel_from_adata(prepared), panel_from_adata(checked))
            for key in before:
                _same_array(before[key], after[key])
        for key, value in prepared.uns.items():
            if isinstance(value, str):
                assert checked.uns[key] == value
        destination.parent.mkdir(parents=True, exist_ok=True)
        os.replace(staging, destination)
    finally:
        ad.settings.allow_write_nullable_strings = previous_setting
        if staging.exists():
            staging.unlink()
    return checked

def _file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def compare_pngs(reference, candidate, *, require_sha=True):
    """Compare decoded RGBA pixels and original file SHA256 without editing either."""
    reference, candidate = (Path(reference), Path(candidate))
    old_sha, new_sha = (_file_sha256(reference), _file_sha256(candidate))
    with PILImage.open(reference) as old, PILImage.open(candidate) as new:
        assert old.format == new.format == 'PNG'
        assert old.size == new.size, (reference.name, candidate.name, old.size, new.size)
        changed, max_change = (0, 0)
        for top in range(0, old.height, 256):
            box = (0, top, old.width, min(top + 256, old.height))
            a = np.asarray(old.crop(box).convert('RGBA'))
            b = np.asarray(new.crop(box).convert('RGBA'))
            changed += int(np.any(a != b, axis=2).sum())
            max_change = max(max_change, int(np.abs(a.astype(np.int16) - b.astype(np.int16)).max()))
        result = dict(reference=str(reference.resolve()), candidate=str(candidate.resolve()), width=old.width, height=old.height, reference_mode=old.mode, candidate_mode=new.mode, reference_sha256=old_sha, candidate_sha256=new_sha, sha256_equal=old_sha == new_sha, changed_pixels=changed, max_channel_change=max_change, pixels_equal=changed == 0)
    assert changed == 0, result
    if require_sha:
        assert old_sha == new_sha, result
    return result

def compare_png_sets(pairs, *, expected_pairs=7, require_sha=True):
    pairs = list(pairs.items()) if isinstance(pairs, Mapping) else list(pairs)
    assert len(pairs) == expected_pairs
    return pd.DataFrame([compare_pngs(old, new, require_sha=require_sha) for old, new in pairs])

In [ ]:
#3. =========== Reviewed membership and common gene metadata
# Readable reviewed metadata externalized without changing the accepted content.
from io import StringIO
import hashlib, json
import numpy as np
import pandas as pd

def _restore_review_frame(tsv_text, schema):
    frame = pd.read_csv(StringIO(tsv_text), sep='\t', index_col=0, dtype=str,
                        keep_default_na=False, na_filter=False, encoding='utf-8')
    assert list(frame.columns) == schema['columns'] and frame.index.is_unique
    frame.index.name = schema['index_name']
    frame.index = frame.index.astype(schema['index_dtype'])
    for col, field in schema['fields'].items():
        s = frame[col]
        missing = s.eq(schema['na_token'])
        dtype = field['dtype']
        if dtype == 'category':
            cats = pd.Index(field['categories'], dtype=field['categories_dtype'])
            lookup = {str(value): value for value in cats}
            assert len(lookup) == len(cats)
            assert s.loc[~missing].isin(lookup).all()
            values = s.map(lookup)
            frame[col] = pd.Categorical(values, categories=cats, ordered=field['ordered'])
        elif dtype == 'object':
            values = s.mask(missing, np.nan).astype(object)
            for kind, rows in field.get('object_scalar_rows', {}).items():
                transform = {'bool': lambda x: x == 'True', 'int': int, 'float': float}[kind]
                values.iloc[rows] = [transform(s.iloc[i]) for i in rows]
            for kind, rows in field.get('object_missing_rows', {}).items():
                values.iloc[rows] = {'None': None, 'pd.NA': pd.NA, 'nan': np.nan}[kind]
            frame[col] = values
        elif dtype in ('bool', 'boolean'):
            values = s.map({'True': True, 'False': False, schema['na_token']: pd.NA})
            frame[col] = values.astype(dtype)
        elif dtype.startswith('string'):
            frame[col] = s.mask(missing, pd.NA).astype(pd.StringDtype(storage=field.get('storage', 'python')))
        else:
            frame[col] = s.mask(missing, np.nan).astype(dtype)
    return frame

# Set metadata/E11_review.json, externalized from Code_def/E11_clustering-2.ipynb cell 3.
REVIEW_FILE = ROOT
# Set metadata/E11_cells.tsv, externalized from the same reviewed source cell.
REVIEW_CELLS_FILE = ROOT
# Set metadata/common_genes.tsv, shared unchanged by the nine reviewed stage sources.
REVIEW_GENES_FILE = ROOT
_review = json.loads(REVIEW_FILE.read_bytes().decode("utf-8"))
_approved_obs_tsv = REVIEW_CELLS_FILE.read_bytes().decode("utf-8")
_common_var_tsv = REVIEW_GENES_FILE.read_bytes().decode("utf-8")
_review_payload = dict(format="reviewed-stage-json-tsv-v1", stage="E11.5",
                       review=_review, approved_obs_tsv=_approved_obs_tsv,
                       common_var_tsv=_common_var_tsv)
_review_bytes = json.dumps(_review_payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
assert hashlib.sha256(_review_bytes).hexdigest() == "367c4479b25ee6d8ae2f9cab6f4fef48b7d2e21a2fd75bce6269e4e6febf934f", "Reviewed metadata differs from the accepted source."
review = _review
approved_obs = _restore_review_frame(_approved_obs_tsv, review["schemas"]["cells"])
common_var = _restore_review_frame(_common_var_tsv, review["schemas"]["genes"])
groups133 = review["marker_groups"]
assert approved_obs.stage_standardized.eq("E11.5").all()
assert approved_obs.index.is_unique and common_var.index.is_unique
assert common_var.shape == (11467, 26)
del _review_payload, _review_bytes, _approved_obs_tsv, _common_var_tsv, _review

groups133 = review['marker_groups']
configure_reviewed_metadata(review['E17_CURATED_LABELS'], review['E9_LOWER_CONFIDENCE_IDS'])
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print({p: version(p) for p in ('scanpy', 'anndata', 'numpy', 'scipy', 'scikit-learn', 'umap-learn', 'igraph')})

def preview(path, width=1200):
    with PILImage.open(path) as picture:
        picture = picture.convert('RGB')
        picture.thumbnail((width, 2200), PILImage.Resampling.LANCZOS)
        buffer = BytesIO()
        picture.save(buffer, format='PNG', optimize=True)
    display(Image(data=buffer.getvalue()))

In [ ]:
#4. =========== Reviewed source pools
STAGE, STAGE_STANDARDIZED = ('E11', 'E11.5')
stage_obs = approved_obs.loc[approved_obs.stage_standardized.eq(STAGE_STANDARDIZED)].copy()
cases = [dict(c) for c in review['SOURCE_CASES'] if c['stage'] == STAGE_STANDARDIZED]
for case in cases:
    if case['study'] == 'GSE195500' and case['normalization'] == 'source_endogenous':
        case['normalization_exclude_hemoglobin'] = True
print('Retained cells:', len(stage_obs))
display(pd.DataFrame([{'source': c['tag'], 'parent_cells': len(c['cells']), 'retained_cells': len(c['reviewed_labels']), 'HVG': c['hvgs'], 'PCA': c['n_pcs'], 'resolution': c['resolution'], 'batch_key': c['harmony_key']} for c in cases]))

In [ ]:
#5. =========== Original count files and full-source QC
raw_parts = {}
for study in dict.fromkeys((c['study'] for c in cases)):
    requested = list(dict.fromkeys((cell for c in cases if c['study'] == study for cell in c['cells'])))
    if study == 'GSE178931':
        libraries = list(dict.fromkeys((cell.split('__')[1] for cell in requested)))
        parts = [read_early_csv(ROOT, library, [cell for cell in requested if cell.split('__')[1] == library]) for library in libraries]
        raw = ad.concat(parts, join='inner', merge='first', uns_merge='unique')
        raw = raw[requested].copy()
        raw.uns['library_qc_json'] = json.dumps([p.uns['raw_source_qc'] for p in parts])
    else:
        raise ValueError(study)
    raw.layers['counts'] = raw.X.copy()
    raw_parts[study] = raw
    print(study, raw.shape, 'integer counts; original source gene axis')

In [ ]:
#6. =========== Count QC and the reviewed parent/retained membership
source_inputs = {}
for case in cases:
    raw = attach_source_review(raw_parts[case['study']], case, approved_obs)
    assert raw.obs.qc_pass.fillna(False).all(), 'A reviewed parent cell fails the original count QC'
    assert raw.obs.retained_in_atlas.sum() == len(case['reviewed_labels'])
    source_inputs[case['tag']] = raw
    print(case['tag'], raw.n_obs, 'parent cells;', int(raw.obs.retained_in_atlas.sum()), 'retained')

In [ ]:
#7. =========== Complete-axis CP10k and log1p
TARGET_SUM = 10000
expressions, panels, preparation = ({}, {}, {})
for case in cases:
    expression, panel, audit = prepare_source_expression(source_inputs[case['tag']], case, common_var, groups133, target_sum=TARGET_SUM)
    expressions[case['tag']], panels[case['tag']], preparation[case['tag']] = (expression, panel, audit)
    print(case['tag'], 'normalization genes:', expression.n_vars)

In [ ]:
#8. =========== Recorded HVG, PCA, source correction and clustering
PCA_PARAMETERS = dict(svd_solver='arpack', random_state=RANDOM_SEED, mask_var=None, scale_clip=10, unit_variance=True)
HVG_PARAMETERS = dict(flavor='seurat', rank_at_least=5000, seurat_v3_span=0.8, excluded_symbol_regex='(?i)^Hb[ab]-')
HARMONY_PARAMETERS = dict(theta=2, lamb=1, sigma=0.1, tau=0, nclust=None, block_size=0.05, max_iter_harmony=30, max_iter_kmeans=20, epsilon_harmony=0.0001, epsilon_cluster=1e-05, random_state=RANDOM_SEED, device='cpu')
GRAPH_PARAMETERS = dict(n_neighbors=15, metric='euclidean', transformer='sklearn', random_state=RANDOM_SEED, flavor='igraph', directed=False, n_iterations=2, min_dist=0.35, spread=1.0)
results, result_panels, audits = ({}, {}, {})
for case in cases:
    hvg_parameters = dict(HVG_PARAMETERS)
    if case.get('author1371'):
        hvg_parameters['excluded_symbols'] = review['author_glial1371']
    for hvg in case['hvgs']:
        graph, audit = fit_source_case(expressions[case['tag']], case, hvg, pca_parameters=PCA_PARAMETERS, graph_parameters=GRAPH_PARAMETERS, hvg_parameters=hvg_parameters, harmony_parameters=HARMONY_PARAMETERS, n_threads=N_THREADS)
        tag = case['tag'] if 'HVG' in case['tag'] else f"{case['tag']}_HVG{hvg}"
        results[tag], result_panels[tag] = (graph, panels[case['tag']])
        audits[tag] = dict(preparation[case['tag']], **audit)
        print(tag, 'Leiden groups:', graph.obs.leiden.nunique())

In [ ]:
#9. =========== Paired UMAP and the complete marker DotPlot
UMAP_POINT_SIZE, UMAP_ALPHA = (4.0, 0.82)
MAX_DOT_AREA, COLUMN_GAP_RATIO = (36, 0.5)
UMAP_STYLE.update(point_size=UMAP_POINT_SIZE, point_alpha=UMAP_ALPHA)
DOTPLOT_STYLE.update(max_dot_area=MAX_DOT_AREA, column_gap_ratio=COLUMN_GAP_RATIO)
figure_paths, dot_statistics, palette = ({}, {}, {})
for tag, graph in results.items():
    # Set the required input, output, or working path for this step.
    out_dir = ROOT
    out_dir.mkdir(parents=True, exist_ok=True)
    paths, stats, colors = render_source_results({tag: graph}, {tag: result_panels[tag]}, {tag: audits[tag]}, out_dir, FIGURE_DPI)
    output_paths = paths[tag]
    reference_prefix = SOURCE_PLOT_PREFIXES.get(tag, tag)
    renamed = dict(legend=output_paths[0], labels=output_paths[1], dotplots=output_paths[2:])
    reference_names = [f'{reference_prefix}_UMAP_legend.png', f'{reference_prefix}_UMAP_labels.png', f'{reference_prefix}_DotPlot_full.png'] + [f'{reference_prefix}_DotPlot_{i}.png' for i in range(1, 5)]
    # Set the required input, output, or working path for this step.
    pairs = [(ROOT, path) for name, path in zip(reference_names, output_paths)]
    matches = compare_png_sets(pairs, expected_pairs=7)
    figure_paths[tag], dot_statistics[tag] = (renamed, stats[tag])
    print(tag, '7 PNGs: exact pixel and SHA256 match')
    preview(renamed['legend'], width=1000)
    preview(renamed['labels'], width=1000)

In [ ]:
#10. =========== Full and split marker panels
for tag, paths in figure_paths.items():
    print(tag)
    for path in paths['dotplots']:
        preview(path)

In [ ]:
#11. =========== Save the retained raw-count integration input
# Set the stage E*_Embryonic_Inte_adata.h5ad output file path, generated by this notebook's final export.
RESULT_FILE = ROOT
stage_input = make_stage_input(raw_parts, stage_obs, common_var, groups133)
stage_input.uns['source_analysis_json'] = provenance_json(audits)
stage_input.uns['figure_paths_json'] = json.dumps(figure_paths)
saved = save_h5ad_verified(stage_input, RESULT_FILE, ROOT, overwrite=True, provenance=dict(stage=STAGE_STANDARDIZED, notebook=f'{STAGE}_clustering-2.ipynb', source_labels_used_for_fit=False, original_counts_unchanged=True))
assert saved.shape == (len(stage_obs), 11467)
assert saved.obs_names.equals(stage_obs.index)
assert not saved.obsp and (not any((key.startswith('X_') for key in saved.obsm)))
print('Saved:', RESULT_FILE, saved.shape)
display(saved.obs.groupby(['study_id', 'cell_type_plot'], observed=True).size().rename('Retained cells'))